# 07 Pydantic Gemini LLM Repetition Seminar

## Pydantic to structure Gemini output

In [54]:
from dotenv import load_dotenv
import os 
from google import genai

load_dotenv()

client= genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
response=client.models.generate_content(model = "gemini-2.5-flash", contents= "tell me a programming joke")
print(response.text)

Why do programmers prefer dark mode?

Because light attracts bugs!

---

*(Another classic:)*

There are 10 types of people in the world: those who understand binary, and those who don't.


In [3]:
response.text

'Why do programmers always mix up Halloween and Christmas?\n\nBecause Oct 31 == Dec 25!'

In [4]:
print(response.text)

Why do programmers always mix up Halloween and Christmas?

Because Oct 31 == Dec 25!


In [5]:
def ask_llm(prompt):
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents= prompt
)
    return response.text

ask_llm("Du är en Göteborgare, ge mig ett skämt som är go' ")

"Hallå där! Klart du ska få ett skämt som är go', du! Här kommer en riktig klassiker, änna:\n\nVarför fick gö-teborgaren inte igång dammsugaren?\n\n... För att sladden var *änna* glapp!\n\nHahaha! Ja, den är allt lite go' den! Tycker jag la'."

## Try to get data from our LLM
- give the LLM:
    - a role
    - a task
    - an example
    - format

In [6]:
response = ask_llm("""
Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
Generera bostadspriser, månadsavgifter, address och stad, boarea i jsonformat (ej markdown)
                   
Exempel: 
                {
                   "address": Fågelvägen 5,
                   "price_sek": 3000000,
                   "monthly_fee": 4000,
                   "city": "Göteborg",
                   "living_area_kvm": 60
                }
    
        Ge mig en lista på 5 bostäder

""")

response

'[\n                {\n                   "address": "Södermannagatan 12",\n                   "price_sek": 5500000,\n                   "monthly_fee": 3200,\n                   "city": "Stockholm",\n                   "living_area_kvm": 55\n                },\n                {\n                   "address": "Linnégatan 34",\n                   "price_sek": 4800000,\n                   "monthly_fee": 4500,\n                   "city": "Göteborg",\n                   "living_area_kvm": 75\n                },\n                {\n                   "address": "Amiralsgatan 77",\n                   "price_sek": 3600000,\n                   "monthly_fee": 5200,\n                   "city": "Malmö",\n                   "living_area_kvm": 85\n                },\n                {\n                   "address": "Svartbäcksgatan 20",\n                   "price_sek": 2400000,\n                   "monthly_fee": 2800,\n                   "city": "Uppsala",\n                   "living_area_kvm": 40\

In [7]:
print(response) # string that looks like a json list

[
                {
                   "address": "Södermannagatan 12",
                   "price_sek": 5500000,
                   "monthly_fee": 3200,
                   "city": "Stockholm",
                   "living_area_kvm": 55
                },
                {
                   "address": "Linnégatan 34",
                   "price_sek": 4800000,
                   "monthly_fee": 4500,
                   "city": "Göteborg",
                   "living_area_kvm": 75
                },
                {
                   "address": "Amiralsgatan 77",
                   "price_sek": 3600000,
                   "monthly_fee": 5200,
                   "city": "Malmö",
                   "living_area_kvm": 85
                },
                {
                   "address": "Svartbäcksgatan 20",
                   "price_sek": 2400000,
                   "monthly_fee": 2800,
                   "city": "Uppsala",
                   "living_area_kvm": 40
                },
         

## Parse and validate data

In [8]:
from pydantic import BaseModel, Field
import json

class Apartment(BaseModel):
    address: str
    price_sek: int = Field(gt=1000000, lt=8000000) #field provides another layer of validation
    monthly_fee: int
    city: str
    living_area_kvm: int

class ApartmentList(BaseModel):
    objects: list[Apartment]
    
apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments #instance of Apartment class containing a list which contains apartment objects

ApartmentList(objects=[Apartment(address='Södermannagatan 12', price_sek=5500000, monthly_fee=3200, city='Stockholm', living_area_kvm=55), Apartment(address='Linnégatan 34', price_sek=4800000, monthly_fee=4500, city='Göteborg', living_area_kvm=75), Apartment(address='Amiralsgatan 77', price_sek=3600000, monthly_fee=5200, city='Malmö', living_area_kvm=85), Apartment(address='Svartbäcksgatan 20', price_sek=2400000, monthly_fee=2800, city='Uppsala', living_area_kvm=40), Apartment(address='Ågatan 80', price_sek=2900000, monthly_fee=3800, city='Linköping', living_area_kvm=65)])

In [9]:
apartments.objects

[Apartment(address='Södermannagatan 12', price_sek=5500000, monthly_fee=3200, city='Stockholm', living_area_kvm=55),
 Apartment(address='Linnégatan 34', price_sek=4800000, monthly_fee=4500, city='Göteborg', living_area_kvm=75),
 Apartment(address='Amiralsgatan 77', price_sek=3600000, monthly_fee=5200, city='Malmö', living_area_kvm=85),
 Apartment(address='Svartbäcksgatan 20', price_sek=2400000, monthly_fee=2800, city='Uppsala', living_area_kvm=40),
 Apartment(address='Ågatan 80', price_sek=2900000, monthly_fee=3800, city='Linköping', living_area_kvm=65)]

In [10]:
apartments.objects[1].address

'Linnégatan 34'

In [11]:
apartments.objects[1].address, apartments.objects[1].city

('Linnégatan 34', 'Göteborg')

In [12]:
addresses = [apartment.address for apartment in apartments.objects ]

addresses

['Södermannagatan 12',
 'Linnégatan 34',
 'Amiralsgatan 77',
 'Svartbäcksgatan 20',
 'Ågatan 80']

In [13]:
# filter by price. list of apartment instances, pick out Field of interest
addresses = [apartment.address for apartment in apartments.objects if apartment.price_sek < 4000000]

addresses

['Amiralsgatan 77', 'Svartbäcksgatan 20', 'Ågatan 80']

In [30]:
# Get address, city, price, monthly_fee for interval 4M - 8M

addresses_4_to_8 = [(apartment.address, apartment.city, apartment.price_sek, apartment.monthly_fee) for apartment in apartments.objects if apartment.price_sek > 4000000 < 8000000]

addresses_4_to_8


[('Södermannagatan 12', 'Stockholm', 5500000, 3200),
 ('Linnégatan 34', 'Göteborg', 4800000, 4500)]

## Load data into duckdb database

In [31]:
filtered_homes = [
    home for home in apartments.objects if 4_000_000 < home.price_sek < 8_000_000
]

filtered_homes

[Apartment(address='Södermannagatan 12', price_sek=5500000, monthly_fee=3200, city='Stockholm', living_area_kvm=55),
 Apartment(address='Linnégatan 34', price_sek=4800000, monthly_fee=4500, city='Göteborg', living_area_kvm=75)]

In [32]:
# convert to df
df_homes_filtered = pd.DataFrame(
    [
        home.model_dump(include={"address", "city", "price_sek", "monthly_fee"})
        for home in filtered_homes
    ]
)
df_homes_filtered

,address,price_sek,monthly_fee,city
0,Södermannagatan 12,5500000,3200,Stockholm
1,Linnégatan 34,4800000,4500,Göteborg


## save to json - serialize pydantic model

In [33]:
apartments

ApartmentList(objects=[Apartment(address='Södermannagatan 12', price_sek=5500000, monthly_fee=3200, city='Stockholm', living_area_kvm=55), Apartment(address='Linnégatan 34', price_sek=4800000, monthly_fee=4500, city='Göteborg', living_area_kvm=75), Apartment(address='Amiralsgatan 77', price_sek=3600000, monthly_fee=5200, city='Malmö', living_area_kvm=85), Apartment(address='Svartbäcksgatan 20', price_sek=2400000, monthly_fee=2800, city='Uppsala', living_area_kvm=40), Apartment(address='Ågatan 80', price_sek=2900000, monthly_fee=3800, city='Linköping', living_area_kvm=65)])

In [34]:
# dictionary
apartments.model_dump()

{'objects': [{'address': 'Södermannagatan 12',
   'price_sek': 5500000,
   'monthly_fee': 3200,
   'city': 'Stockholm',
   'living_area_kvm': 55},
  {'address': 'Linnégatan 34',
   'price_sek': 4800000,
   'monthly_fee': 4500,
   'city': 'Göteborg',
   'living_area_kvm': 75},
  {'address': 'Amiralsgatan 77',
   'price_sek': 3600000,
   'monthly_fee': 5200,
   'city': 'Malmö',
   'living_area_kvm': 85},
  {'address': 'Svartbäcksgatan 20',
   'price_sek': 2400000,
   'monthly_fee': 2800,
   'city': 'Uppsala',
   'living_area_kvm': 40},
  {'address': 'Ågatan 80',
   'price_sek': 2900000,
   'monthly_fee': 3800,
   'city': 'Linköping',
   'living_area_kvm': 65}]}

In [35]:
# str of json data
apartments.model_dump_json()

'{"objects":[{"address":"Södermannagatan 12","price_sek":5500000,"monthly_fee":3200,"city":"Stockholm","living_area_kvm":55},{"address":"Linnégatan 34","price_sek":4800000,"monthly_fee":4500,"city":"Göteborg","living_area_kvm":75},{"address":"Amiralsgatan 77","price_sek":3600000,"monthly_fee":5200,"city":"Malmö","living_area_kvm":85},{"address":"Svartbäcksgatan 20","price_sek":2400000,"monthly_fee":2800,"city":"Uppsala","living_area_kvm":40},{"address":"Ågatan 80","price_sek":2900000,"monthly_fee":3800,"city":"Linköping","living_area_kvm":65}]}'

In [ ]:
with open("apartments.json", "w") as json_file:
    json_file.write(apartments.model_dump_json(indent=3))

## Pandas dataframe alternative way

In [45]:
apartments.objects

[Apartment(address='Södermannagatan 12', price_sek=5500000, monthly_fee=3200, city='Stockholm', living_area_kvm=55),
 Apartment(address='Linnégatan 34', price_sek=4800000, monthly_fee=4500, city='Göteborg', living_area_kvm=75),
 Apartment(address='Amiralsgatan 77', price_sek=3600000, monthly_fee=5200, city='Malmö', living_area_kvm=85),
 Apartment(address='Svartbäcksgatan 20', price_sek=2400000, monthly_fee=2800, city='Uppsala', living_area_kvm=40),
 Apartment(address='Ågatan 80', price_sek=2900000, monthly_fee=3800, city='Linköping', living_area_kvm=65)]

In [49]:
addresses = [apartment.address for apartment in apartments.objects]
prices = [apartment.price_sek for apartment in apartments.objects]
monthly_fees = [apartment.monthly_fee for apartment in apartments.objects]
areas = [apartment.living_area_kvm for apartment in apartments.objects]

df = pd.DataFrame(
    {"address": addresses, "living_area_kvm": areas, "price": prices, "monthly_fee": monthly_fees}
)

df

,address,living_area_kvm,price,monthly_fee
0,Södermannagatan 12,55,5500000,3200
1,Linnégatan 34,75,4800000,4500
2,Amiralsgatan 77,85,3600000,5200
3,Svartbäcksgatan 20,40,2400000,2800
4,Ågatan 80,65,2900000,3800


In [50]:
df

,address,living_area_kvm,price,monthly_fee
0,Södermannagatan 12,55,5500000,3200
1,Linnégatan 34,75,4800000,4500
2,Amiralsgatan 77,85,3600000,5200
3,Svartbäcksgatan 20,40,2400000,2800
4,Ågatan 80,65,2900000,3800


In [51]:
df.to_dict()

{'address': {0: 'Södermannagatan 12',
  1: 'Linnégatan 34',
  2: 'Amiralsgatan 77',
  3: 'Svartbäcksgatan 20',
  4: 'Ågatan 80'},
 'living_area_kvm': {0: 55, 1: 75, 2: 85, 3: 40, 4: 65},
 'price': {0: 5500000, 1: 4800000, 2: 3600000, 3: 2400000, 4: 2900000},
 'monthly_fee': {0: 3200, 1: 4500, 2: 5200, 3: 2800, 4: 3800}}

In [52]:
df.to_dict(orient="records")

[{'address': 'Södermannagatan 12',
  'living_area_kvm': 55,
  'price': 5500000,
  'monthly_fee': 3200},
 {'address': 'Linnégatan 34',
  'living_area_kvm': 75,
  'price': 4800000,
  'monthly_fee': 4500},
 {'address': 'Amiralsgatan 77',
  'living_area_kvm': 85,
  'price': 3600000,
  'monthly_fee': 5200},
 {'address': 'Svartbäcksgatan 20',
  'living_area_kvm': 40,
  'price': 2400000,
  'monthly_fee': 2800},
 {'address': 'Ågatan 80',
  'living_area_kvm': 65,
  'price': 2900000,
  'monthly_fee': 3800}]

In [48]:
import pandas as pd

addresses = [apartment.address for apartment in apartments.objects]
prices= [apartment.price_sek for apartment in apartments.objects]
monthly_fees = [apartment.monthly_fee for apartment in apartments.objects]
living_areas = [apartment.living_area_kvm for apartment in apartments.objects]

pd.DataFrame({"address": addresses, "living_area_kvm": living_areas, "price_sek": prices, "monthly_fee": monthly_fees})

,address,living_area_kvm,price_sek,monthly_fee
0,Södermannagatan 12,55,5500000,3200
1,Linnégatan 34,75,4800000,4500
2,Amiralsgatan 77,85,3600000,5200
3,Svartbäcksgatan 20,40,2400000,2800
4,Ågatan 80,65,2900000,3800


## put this data into a duckdb database

Approach 1
- duckdb read_csv_auto...

Approach 2
- open up a connection to duckdb
- create schema
- create tables from df

Approach 3 (naive)
- create tables
- insert data manually (or parse with python)

Approach 4 - dlt
- extract and load df into duckdb

## load data into duckdb using dlt

In [44]:
# dlt - data load tool
import dlt
import duckdb

@dlt.resource(write_disposition="replace", table_name="apartment")
def load_data():
    for record in df.to_dict(orient="records"):
        yield record

pipeline = dlt.pipeline(
    pipeline_name="apartments",
    destination="duckdb", 
    dataset_name="staging",
)

load_info = pipeline.run(load_data())
print(load_info)

Pipeline apartments load step completed in 0.08 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:////Users/susannarokka/Desktop/NBI_year_2/Repo/AI_Engineering_OPA24_Susanna_Rokka/code-alongs/07_pydantic/apartments.duckdb location to store data
Load package 1757331616.640169 is LOADED and contains no failed jobs


In [53]:
df

,address,living_area_kvm,price,monthly_fee
0,Södermannagatan 12,55,5500000,3200
1,Linnégatan 34,75,4800000,4500
2,Amiralsgatan 77,85,3600000,5200
3,Svartbäcksgatan 20,40,2400000,2800
4,Ågatan 80,65,2900000,3800


In [42]:
df.to_dict(orient="records")

[{0: 'Amiralsgatan 77'}, {0: 'Svartbäcksgatan 20'}, {0: 'Ågatan 80'}]

In [19]:
addresses_4_to_8 = [
    ["address", "city", "price_sek", "monthly_fee"]
    for apartment
    in apartments.objects
    if 4000000 < apartment.price_sek < 8000000
]

addresses_4_to_8

[['address', 'city', 'price_sek', 'monthly_fee'],
 ['address', 'city', 'price_sek', 'monthly_fee']]

## Save to json - serialize pydantic model

In [23]:
apartments

ApartmentList(objects=[Apartment(address='Södermannagatan 12', price_sek=5500000, monthly_fee=3200, city='Stockholm', living_area_kvm=55), Apartment(address='Linnégatan 34', price_sek=4800000, monthly_fee=4500, city='Göteborg', living_area_kvm=75), Apartment(address='Amiralsgatan 77', price_sek=3600000, monthly_fee=5200, city='Malmö', living_area_kvm=85), Apartment(address='Svartbäcksgatan 20', price_sek=2400000, monthly_fee=2800, city='Uppsala', living_area_kvm=40), Apartment(address='Ågatan 80', price_sek=2900000, monthly_fee=3800, city='Linköping', living_area_kvm=65)])

In [24]:
apartments.model_dump() # list of dictionaries

{'objects': [{'address': 'Södermannagatan 12',
   'price_sek': 5500000,
   'monthly_fee': 3200,
   'city': 'Stockholm',
   'living_area_kvm': 55},
  {'address': 'Linnégatan 34',
   'price_sek': 4800000,
   'monthly_fee': 4500,
   'city': 'Göteborg',
   'living_area_kvm': 75},
  {'address': 'Amiralsgatan 77',
   'price_sek': 3600000,
   'monthly_fee': 5200,
   'city': 'Malmö',
   'living_area_kvm': 85},
  {'address': 'Svartbäcksgatan 20',
   'price_sek': 2400000,
   'monthly_fee': 2800,
   'city': 'Uppsala',
   'living_area_kvm': 40},
  {'address': 'Ågatan 80',
   'price_sek': 2900000,
   'monthly_fee': 3800,
   'city': 'Linköping',
   'living_area_kvm': 65}]}

In [25]:
apartments.model_dump_json() # gives string of json data

'{"objects":[{"address":"Södermannagatan 12","price_sek":5500000,"monthly_fee":3200,"city":"Stockholm","living_area_kvm":55},{"address":"Linnégatan 34","price_sek":4800000,"monthly_fee":4500,"city":"Göteborg","living_area_kvm":75},{"address":"Amiralsgatan 77","price_sek":3600000,"monthly_fee":5200,"city":"Malmö","living_area_kvm":85},{"address":"Svartbäcksgatan 20","price_sek":2400000,"monthly_fee":2800,"city":"Uppsala","living_area_kvm":40},{"address":"Ågatan 80","price_sek":2900000,"monthly_fee":3800,"city":"Linköping","living_area_kvm":65}]}'

In [29]:
with open ("apartments.json", "w") as json_file:
           json_file.write(apartments.model_dump_json(indent=3))